In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("task4-analytics")
         .enableHiveSupport()
         .getOrCreate())

DB = "project"

In [ ]:
q1_sql = f"""
WITH base AS (
  SELECT ticker,
         to_date(trade_date)    AS d,
         CAST(high AS DOUBLE)   AS h,
         CAST(low AS DOUBLE)    AS l,
         CAST(close AS DOUBLE)  AS c,
         CAST(volume AS DOUBLE) AS v
  FROM {DB}.prices_cleaned
  WHERE status = 'accepted'
),
ratio AS (
  SELECT ticker, d, (h - l) / c AS day_range,
         v / AVG(v) OVER (PARTITION BY ticker ORDER BY d
                          ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING) AS vol_ratio,
         COUNT(*) OVER (PARTITION BY ticker ORDER BY d
                        ROWS BETWEEN 20 PRECEDING AND 1 PRECEDING) AS n_hist
  FROM base
)
SELECT CASE WHEN vol_ratio < 1.5 THEN '1_normal (<1.5x)'
            WHEN vol_ratio < 2.5 THEN '2_high (1.5-2.5x)'
            ELSE '3_very_high (>2.5x)' END AS bucket,
       COUNT(*)                          AS n_rows,
       AVG(day_range)                    AS avg_day_range,
       percentile_approx(day_range, 0.5) AS median_day_range
FROM ratio
WHERE n_hist = 20 AND vol_ratio IS NOT NULL
GROUP BY CASE WHEN vol_ratio < 1.5 THEN '1_normal (<1.5x)'
              WHEN vol_ratio < 2.5 THEN '2_high (1.5-2.5x)'
              ELSE '3_very_high (>2.5x)' END
ORDER BY bucket
"""

q1_result = spark.sql(q1_sql)
q1_result.show(truncate=False)

In [ ]:
import os
import matplotlib.pyplot as plt

os.makedirs("task4/outputs", exist_ok=True)

# The result has only 3 rows, so it is safe to turn into pandas for plotting
q1_pdf = q1_result.toPandas()
q1_pdf.to_csv("task4/outputs/q1_volume_buckets.csv", index=False)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(q1_pdf["bucket"], q1_pdf["avg_day_range"] * 100)

# Write the group size on top of each bar
for bar, n in zip(bars, q1_pdf["n_rows"]):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
            f"n = {n:,}", ha="center", va="bottom")

ax.set_xlabel("Volume vs the stock's own previous 20-day average")
ax.set_ylabel("Average daily range, (high - low) / close (%)")
ax.set_title("Q1: Daily price range by volume bucket")
plt.tight_layout()
plt.savefig("task4/outputs/q1_chart.png", dpi=150)
plt.show()